# NSTT — SLR54 Data Preparation (T-002)

Downloads [OpenSLR SLR54](https://www.openslr.org/54/), preprocesses audio (16 kHz mono, duration filter), NFC-normalizes transcripts, and produces speaker-disjoint 80/10/10 manifests.

Prerequisites: Run `00_setup.ipynb` first. Store the project on Google Drive so raw/processed audio persist across sessions.

This notebook is resumable: if Colab disconnects mid-run, just re-run all cells — Cell 8 will skip everything already processed and continue from where it left off.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
import sys, os, subprocess
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/NSTT")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"{PROJECT_ROOT} not found — repo missing from Drive.")

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

install_result = subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], capture_output=True, text=True)
if install_result.returncode != 0:
    print(install_result.stderr[-2000:])
    raise RuntimeError("pip install failed")
print("✅ Dependencies installed.")

In [ ]:
# Optional: limit to one zip for a quick smoke test; set to None for full corpus (~8 GB)
ZIP_SUBSET = None  # e.g. ["asr_nepali_0.zip"] for smoke test
SEED = 42
MAX_UTTERANCES = None  # None = process all utterances
raw_dir = PROJECT_ROOT / "data" / "raw" / "slr54"

In [ ]:
# Pre-flight: verify any already-downloaded zips aren't corrupted/partial
expected_min_mb = 50

if not raw_dir.exists():
    print("No downloads yet — will start fresh.")
else:
    suspicious = []
    for f in sorted(raw_dir.glob("*.zip")):
        size_mb = f.stat().st_size / (1024 * 1024)
        flag = "⚠️ SUSPICIOUS" if size_mb < expected_min_mb else "✅"
        print(f"{f.name}: {size_mb:.1f} MB  {flag}")
        if size_mb < expected_min_mb:
            suspicious.append(f)
    if suspicious:
        for f in suspicious:
            f.unlink()
            print(f"removed {f.name}")
    else:
        print("✅ All existing downloads look complete.")

In [ ]:
from src.slr54 import download_tsv, download_zips

tsv_path = download_tsv(raw_dir)
print(f"✅ TSV ready: {tsv_path}")

zip_paths = download_zips(raw_dir, zip_names=ZIP_SUBSET)
print(f"✅ {len(zip_paths)} zip(s) ready.")

In [ ]:
from src.slr54 import extract_zips

extract_dir = raw_dir / "extracted"
extract_zips(raw_dir, zip_paths, extract_dir)
print(f"✅ Extraction done: {extract_dir}")

In [ ]:
from src.slr54 import parse_utt_spk_text_tsv, index_audio_files, attach_audio_paths

utterances = parse_utt_spk_text_tsv(tsv_path)
print(f"✅ Parsed {len(utterances)} utterances.")

audio_index = index_audio_files(extract_dir)
print(f"✅ Indexed {len(audio_index)} audio files.")

utterances = attach_audio_paths(utterances, audio_index)
if MAX_UTTERANCES is not None:
    utterances = utterances[:MAX_UTTERANCES]
print(f"Utterances ready: {len(utterances)}")

In [ ]:
# Process audio: batched, crash-safe, resumable.
# Processes to local Colab disk (fast — avoids slow Drive I/O per-file), then flushes
# each batch to Drive immediately so a disconnect only costs one batch, not the whole run.
from tqdm.notebook import tqdm
from src.preprocessing import process_utterance
import json, shutil

processed_dir_local = Path("/content/processed_local")
processed_dir_local.mkdir(parents=True, exist_ok=True)

processed_dir_drive = PROJECT_ROOT / "data" / "processed"
processed_dir_drive.mkdir(parents=True, exist_ok=True)

progress_file = PROJECT_ROOT / "data" / "manifests" / "_progress.jsonl"
progress_file.parent.mkdir(parents=True, exist_ok=True)

# --- Resume from Drive progress file ---
done_ids = set()
processed_records = []
if progress_file.exists():
    with open(progress_file) as f:
        for line in f:
            row = json.loads(line)
            processed_records.append(row)
            done_ids.add(row["utterance_id"])
    print(f"↩️  Resuming — {len(done_ids)} utterances already done, skipping them.")

remaining = [u for u in utterances if u.utterance_id not in done_ids]
print(f"Remaining to process: {len(remaining)} / {len(utterances)}")

skipped_no_audio = 0
skipped_duration = 0
BATCH_SIZE = 2000
batch_buffer = []

def flush_batch(batch_buffer):
    for row in batch_buffer:
        src = processed_dir_local / f"{row['utterance_id']}.wav"
        dst = processed_dir_drive / f"{row['utterance_id']}.wav"
        if src.exists():
            shutil.copy2(src, dst)
    with open(progress_file, "a") as f:
        for row in batch_buffer:
            f.write(json.dumps(row) + "\n")
    print(f"  💾 Flushed {len(batch_buffer)} records to Drive (safe now).")

for utt in tqdm(remaining, desc="Processing utterances"):
    if utt.audio_path is None:
        skipped_no_audio += 1
        continue
    row = process_utterance(
        utterance_id=utt.utterance_id,
        speaker_id=utt.speaker_id,
        transcript=utt.transcript,
        input_audio=utt.audio_path,
        processed_audio_dir=processed_dir_local,
    )
    if row is None:
        skipped_duration += 1
        continue
    row["audio_path"] = str(Path("data/processed") / f"{row['utterance_id']}.wav")
    batch_buffer.append(row)
    processed_records.append(row)

    if len(batch_buffer) >= BATCH_SIZE:
        flush_batch(batch_buffer)
        batch_buffer = []

if batch_buffer:
    flush_batch(batch_buffer)

print(f"✅ Total processed: {len(processed_records)}, skipped (no audio): {skipped_no_audio}, skipped (duration): {skipped_duration}")

In [ ]:
from src.splits import speaker_disjoint_split, count_speaker_overlap
from src.manifests import write_split_manifests

manifest_dir = PROJECT_ROOT / "data" / "manifests"

splits = speaker_disjoint_split(processed_records, seed=SEED)
manifest_paths = write_split_manifests(splits, manifest_dir)
overlap = count_speaker_overlap(splits)

stats = {
    "utterances_in_tsv": len(utterances),
    "processed_after_filter": len(processed_records),
    "train_count": len(splits["train"]),
    "val_count": len(splits["val"]),
    "test_count": len(splits["test"]),
    "speaker_overlap": overlap,
    "seed": SEED,
}
print("✅ Pipeline stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")
print("Manifests written to:", manifest_paths)

In [ ]:
# Spot-check a few manifest rows and processed audio files
import soundfile as sf
from src.manifests import read_jsonl_manifest

for split in ["train", "val", "test"]:
    rows = read_jsonl_manifest(manifest_dir / f"{split}.jsonl")
    print(f"{split}: {len(rows)} utterances")
    for row in rows[:2]:
        audio = PROJECT_ROOT / row["audio_path"]
        data, sr = sf.read(audio)
        ch = 1 if data.ndim == 1 else data.shape[1]
        print(f"  {row['utterance_id']}: sr={sr}, channels={ch}, dur={row['duration_s']}s")